In [ ]:
%%time

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import find_peaks

# Lorenz equilibrium bifurcation diagram

# Classical Lorenz parameters
sigma = 10
beta = 8/3

rho = np.linspace(0, 40, 1000)
z0 = np.zeros_like(rho)
mask = rho > 1

x_plus = np.zeros_like(rho)
x_minus = np.zeros_like(rho)
z_branch = np.zeros_like(rho)

x_plus[mask] = np.sqrt(beta * (rho[mask] - 1))
x_minus[mask] = -np.sqrt(beta * (rho[mask] - 1))
z_branch[mask] = rho[mask] - 1

plt.figure()
plt.plot(rho, z0, 'k', label="Origin")
plt.plot(rho[mask], z_branch[mask], 'b', label="Non-zero equilibria")

plt.axvline(1, linestyle='--', label="Pitchfork bifurcation")

plt.xlabel("rho")
plt.ylabel("z equilibrium value")
plt.title("Equilibrium Diagram of the Lorenz System")
plt.legend()
plt.grid(True)

In [ ]:
%%time
# Discrete Lorenz bifurcation diagram
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import find_peaks

# Parameters
sigma = 10
beta = 8/3

def lorenz(t, state, rho):
    x, y, z = state
    dx = sigma*(y - x)
    dy = x*(rho - z) - y
    dz = x*y - beta*z
    return [dx, dy, dz]

rho_values = np.linspace(0, 40, 400) # rho drives chaos.

rho_plot = []
z_plot = []

for rho in rho_values:
    sol = solve_ivp(
        lorenz,
        [0, 100],
        [1, 1, 1],
        args=(rho,),
        t_eval=np.linspace(0, 100, 8000)
    )

    z = sol.y[2]
    z = z[4000:]  # remove transient

    rho_plot.extend([rho]*len(z[-200:]))
    z_plot.extend(z[-200:])

plt.figure(figsize=(10,6))

# Scatter bifurcation points
plt.plot(rho_plot, z_plot, ',k', alpha=0.25)


plt.axvline(1, linestyle='--',color='r')
plt.axvline(24.74, linestyle='--',color='g')
plt.axvline(28, linestyle='--')

plt.text(0.3, 5, "Stable Fixed Point\nE₀", rotation=90)
plt.text(8, 35, "Stable Non-zero\nEquilibria E±")
plt.text(25, 40, "Hopf\nBifurcation", rotation=90)
plt.text(30, 35, "Fully Chaotic\nStrange Attractor")

roughpoints = {
    1: 'ρ = 1',
    24.74: 'ρ = 24.74',
    28: 'ρ = 28'
}
roughpoints1 = {
    1: 'I',
    24.74: 'I',
    28: 'I'
}


for x, label in roughpoints.items():
    plt.text(x, -3, label, ha='right', va='top', fontsize=10, color='blue')

for x, label in roughpoints1.items():
    plt.text(x, 0, label, ha='right', va='top', fontsize=10, color='blue')

plt.xlabel("ρ")
plt.ylabel("z (long-term values)")
plt.title("Annotated Lorenz Bifurcation Diagram")
plt.ylim(0, 50)
plt.grid(True)

In [ ]:
%%time
#change sigma. sigma controls velocity response to temperature.

beta = 8/3

def lorenz(t, state, sigma, rho):
    x, y, z = state
    dx = sigma*(y-x)
    dy = x*(rho-z) - y
    dz = x*y - beta*z
    return [dx, dy, dz]


rho_values = np.linspace(0,40,250)

sigma_list = [5,10,15,20]

fig, axes = plt.subplots(2,2,figsize=(14,10),sharex=True,sharey=True)
axes = axes.flatten()

for ax, sigma in zip(axes, sigma_list):

    rho_plot = []
    z_plot = []

    for rho in rho_values:

        sol = solve_ivp(
            lorenz,
            [0,80],
            [1,1,1],
            args=(sigma,rho),
            max_step=0.05
        )

        z = sol.y[2]

        # remove transient
        z = z[int(len(z)*0.6):]

        # find local maxima
        peaks,_ = find_peaks(z)

        zmax = z[peaks]

        rho_plot.extend([rho]*len(zmax))
        z_plot.extend(zmax)

    ax.plot(rho_plot,z_plot,'.k',markersize=1)

    ax.axvline(1,linestyle='--',color='r',label='ρ=1 pitchfork bifurcation')
    ax.axvline(24.74,linestyle='--',color='g',label='ρ=24.74 Hopf bifurcation')
    ax.axvline(28,linestyle='--',label='ρ=28 fully developed chaos')

    ax.set_title(f"σ = {sigma}")
    ax.grid(True)

fig.supxlabel("ρ")
fig.supylabel("z maxima")
fig.suptitle("Lorenz Bifurcation Diagram Comparison (Different σ)")

plt.tight_layout()
plt.grid(True)
plt.legend(loc='upper left')

In [ ]:
%%time
#change beta. beta controls damping.

sigma = 10

def lorenz(t, state, sigma, beta, rho):
    x, y, z = state
    return [
        sigma*(y-x),
        x*(rho-z) - y,
        x*y - beta*z
    ]

rho_values = np.linspace(0,40,120)

beta_list = [1, 2, 8/3, 4]

fig, axes = plt.subplots(2,2, figsize=(12,8), sharex=True, sharey=True)
axes = axes.flatten()

for ax, beta in zip(axes, beta_list):

    rho_plot = []
    z_plot = []

    for rho in rho_values:

        sol = solve_ivp(
            lorenz,
            [0,50],
            [1,1,1],
            args=(sigma,beta,rho),
            max_step=0.2
        )

        z = sol.y[2]
        z = z[len(z)//2:]  # remove transient
        z_sample = z[-20:] # keep only final points

        rho_plot.extend([rho]*len(z_sample))
        z_plot.extend(z_sample)

    ax.plot(rho_plot, z_plot, ',k', alpha=0.4)

    ax.axvline(1, linestyle='--', color='r')
    ax.axvline(24.74, linestyle='--', color='g')
    ax.axvline(28, linestyle='--')

    ax.set_title(f"β = {beta:.2f}")
    ax.set_ylim(0,50)
    ax.grid(True)

fig.supxlabel("ρ")
fig.supylabel("z (long-term values)")
fig.suptitle("Lorenz Bifurcation Diagrams for Different β")

plt.tight_layout()
plt.show()